In [7]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import matplotlib.cm as cm
from matplotlib.patches import Patch

In [8]:
cwd = os.getcwd()
cwd

'/Users/ivanatang/Developer/biosensors'

In [9]:
def get_data(file):
    x_data = []
    y_data = []
    
    with open(file, 'r') as f:
        for line in f:
            if not line.startswith(('@', '#')):  # Skip header lines
                parts = line.strip().split()
                if len(parts) >= 2:  # Ensure there are at least two columns of data
                    try:
                        x_data.append(float(parts[0]))
                        y_data.append(float(parts[1]))
                    except ValueError:
                        # Handle cases where data might not be purely numerical
                        continue
    return np.array(x_data), np.array(y_data)

In [10]:
def summary_stats(data):  
    
    data_mean = np.mean(data)
    data_std = np.std(data)
    data_min = np.min(data)
    data_max = np.max(data)
    
    return data_mean, data_std, data_min, data_max

In [ ]:
# Load sequences
base   = "/Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models"
runrel = "prod_md_0p9_cutoff_3dt_64x1_16PME_642dd"

TYPE_SUBDIR = {
    "binder":    "binders",
    "nb":        "nonbinders",
    "low_pkt":   "neg_low_pkt",
    "fail_gate": "neg_fail_gate",
}

GROUP_LABEL = {
    "binder":    "Binder",
    "nb":        "False Positive",
    "low_pkt":   "Low Confidence",
    "fail_gate": "Fail Geometry",
}

GROUP_COLOR = {
    'Binder':         '#648FFF',   # blue
    'False Positive': '#DC267F',   # magenta
    'Low Confidence': '#FE6100',   # orange
    'Fail Geometry':  '#FFB000',   # gold
}

# Legacy seqID-style
legacy_binder_ids = ["seq16", "seq14", "seq10", "seq17", "seq39", "seq42"]
legacy_nb_ids     = ["seq1",  "seq2",  "seq3",  "seq4",  "seq5",  "seq6",  "seq7"]

# New pair_XXXX-style
pair_binder_ids =  ["3069", "3070", "3074", "3092", "3098",
                    "3061", "3062", "3063", "3066", "3067",
                    "3068", "3071", "3072", "3073", "3075",
                    "3076", "3077", "3078", "3079", "3080",
                    "3085", "3086", "3087", "3088", "3091",
                    "3099", "3100", "3101", "3103", "3104", "3108"]
pair_nb_ids      = ["0777", "0827", "1939", "1824",
                    "0052", "0091", "0246", "0269", "0272",
                    "0276", "0288", "0341", "0351", "0450",
                    "0502", "0506", "0549", "0567", "0584",
                    "0602", "0647", "0684", "0736", "0748",
                    "0765", "0776", "0782", "0811", "0822",
                    "0865", "0867", "0868", "0876", "0878",
                    "0883", "0891", "0908", "0932", "0940",
                    "0974", "0977", "1011", "1023", "1102",
                    "1313", "1397", "1654", "1656", "1658",
                    "1671", "1679", "1709", "1710", "1737",
                    "1758", "1784", "1785", "1792", "1835",
                    "1847", "1860", "1914", "1922", "1938",
                    "1961", "1965", "3059"]
pair_low_pkt_ids     = ["0008", "0393", "0482", "1331", "1931",
                        "0065", "0947", "0948", "1413"]
pair_fail_gate_ids   = ["0715", "1094", "1682", "1739", "1941",
                        "0014", "0263", "0540", "0997", "1745"]

# New bind_XXX / nonb_XXX-style
bind_binder_ids = ["019", "020", "022", "034", "041", 
                   "043", "069", "076", "080", "083", 
                   "088", "099", "100", "101", "102", "110"]
nonb_nb_ids     = ["006", "008", "009", "010", "011", 
                   "013", "014", "017", "018", "019", 
                   "021", "022", "023", "024", "025", 
                   "026", "029", "032", "046", "047", 
                   "048", "049", "050", "051", "052", 
                   "053", "054", "056", "066", "067", 
                   "068", "071", "072", "073", "074", 
                   "075", "080", "081", "082", "085",
                   "086", "087", "088", "089", "115", 
                   "116", "117"]

# Each entry: (folder_name, seq_type)
binder_list = (
    [(f"{sid}_binder",       "binder") for sid in legacy_binder_ids] +
    [(f"pair_{i}_binder",    "binder") for i in pair_binder_ids]     +
    [(f"bind_{i}_binder",    "binder") for i in bind_binder_ids]
)

nonbinder_list = (
    [(f"{sid}_nb",           "nb")        for sid in legacy_nb_ids]       +
    [(f"pair_{i}_nb",        "nb")        for i in pair_nb_ids]           +
    [(f"nonb_{i}_nb",        "nb")        for i in nonb_nb_ids]           +
    [(f"pair_{i}_low_pkt",   "low_pkt")   for i in pair_low_pkt_ids]      +
    [(f"pair_{i}_fail_gate", "fail_gate") for i in pair_fail_gate_ids]
)

all_systems = binder_list + nonbinder_list   # (folder_name, seq_type)

GROUP_ORDER = ["Binder", "False Positive", "Low Confidence", "Fail Geometry"]
NM_TO_ANG = 10.0

# -------------------------------------------------------------------
# Some sequences were reprocessed with the updated post-processing
# pipeline, which writes RMSF outputs one level deeper, under a
# "500ns" subdirectory of runrel instead of directly in runrel.
# This applies to all bind_XXX / nonb_XXX sequences, plus a handful
# of pair_XXXX sequences that were resubmitted with the new pipeline.
# -------------------------------------------------------------------
NEW_STRUCTURE_PAIR_IDS = {
    "3061", "3078", "3080",                                   # binder
    "0091", "0272", "0482", "0506", "0567",
    "0736", "0782", "0811", "0822", "1654",                   # nonbinder
}

def rmsf_run_dir(folder_name, seq_type):
    """Directory containing a sequence's rmsf_*.xvg outputs, accounting
    for the newer pipeline layout (RMSF files under runrel/500ns/)."""
    run_dir = os.path.join(base, TYPE_SUBDIR[seq_type], folder_name, runrel)
    parts  = folder_name.split("_")
    prefix = parts[0]
    uses_new_structure = (
        prefix in ("bind", "nonb") or
        (prefix == "pair" and len(parts) >= 2 and parts[1] in NEW_STRUCTURE_PAIR_IDS)
    )
    return os.path.join(run_dir, "500ns") if uses_new_structure else run_dir

In [ ]:

# ------------------------------------------------------------
# Plot average per-residue RMSF for binders vs nonbinders
# ------------------------------------------------------------

# -------------------------------------------------------------------
# Load per-residue RMSF for a list of (folder_name, seq_type) tuples
# Returns a dict: {residue_id: [rmsf_seq1, rmsf_seq2, ...]} in Å
# -------------------------------------------------------------------
def load_rmsf_group(seq_list):
    group = {}
    for folder_name, seq_type in seq_list:
        filepath = os.path.join(rmsf_run_dir(folder_name, seq_type), "rmsf_PL.xvg")
        if not os.path.exists(filepath):
            print(f"WARNING: not found — {filepath}")
            continue
        _, rmsf_nm = get_data(filepath)
        # get_data returns arrays indexed from 0; we need residue IDs from col0
        x_data, y_data = get_data(filepath)
        for resid, rmsf in zip(x_data.astype(int), y_data):
            group.setdefault(resid, []).append(rmsf * NM_TO_ANG)
    return group

binder_data    = load_rmsf_group(binder_list)
nonbinder_data = load_rmsf_group(nonbinder_list)

print(f"Binder sequences loaded:    {len(binder_list)}")
print(f"Nonbinder sequences loaded: {len(nonbinder_list)}")

# -------------------------------------------------------------------
# Compute per-residue mean and SD for residues present in both groups
# -------------------------------------------------------------------
common_residues = sorted(set(binder_data.keys()) & set(nonbinder_data.keys()))

rows = []
for resid in common_residues:
    b_vals  = binder_data[resid]
    nb_vals = nonbinder_data[resid]

    b_mean  = np.mean(b_vals)
    b_sd    = np.std(b_vals,  ddof=1) if len(b_vals)  > 1 else 0.0
    nb_mean = np.mean(nb_vals)
    nb_sd   = np.std(nb_vals, ddof=1) if len(nb_vals) > 1 else 0.0
    pct_diff = (b_mean - nb_mean) / nb_mean * 100 if nb_mean != 0 else np.nan

    rows.append({
        "Residue":           resid,
        "Binder mean (A)":   round(b_mean,           6),
        "Binder SD (A)":     round(b_sd,             6),
        "Nonbinder mean (A)":round(nb_mean,           6),
        "Nonbinder SD (A)":  round(nb_sd,             6),
        "Diff (B - NB, A)":  round(b_mean - nb_mean,  6),
        "% diff (vs NB)":    round(pct_diff,           4),
    })

df = pd.DataFrame(rows).set_index("Residue")
print(f"\nResidues in common: {len(common_residues)}")
print(df.to_string())

# -------------------------------------------------------------------
# Plot
# -------------------------------------------------------------------
residues  = np.array(common_residues)
b_means   = df["Binder mean (A)"].values
b_sds     = df["Binder SD (A)"].values
nb_means  = df["Nonbinder mean (A)"].values
nb_sds    = df["Nonbinder SD (A)"].values
pct_diffs = df["% diff (vs NB)"].values

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 8), dpi=150,
                                constrained_layout=True)

# ---- Panel 1: mean RMSF with shaded SD ----
ax1.plot(residues, b_means,  color="#648FFF", lw=1.5, label=f"Binders (n={len(binder_list)}, mean)")
ax1.fill_between(residues, b_means - b_sds, b_means + b_sds, color="#648FFF", alpha=0.2, label="Binders ± SD")

ax1.plot(residues, nb_means, color="#DC267F", lw=1.5, label=f"Nonbinders (n={len(nonbinder_list)}, mean)")
ax1.fill_between(residues, nb_means - nb_sds, nb_means + nb_sds, color="#DC267F", alpha=0.2, label="Nonbinders ± SD")

ax1.axvspan(84,  90,  color="grey", alpha=0.3, label="Gate (r84–90)")
ax1.axvspan(114, 118, color="grey", alpha=0.3, label="Latch (r114–118)")

ax1.set_xlabel("Residue number", fontsize=9)
ax1.set_ylabel("RMSF (Å)", fontsize=9)
ax1.set_title("Per-residue Cα RMSF: binders vs nonbinders", fontsize=10)
ax1.legend(fontsize=7, ncol=3)
ax1.grid(True, alpha=0.4)

# ---- Panel 2: % difference ----
colors = np.where(pct_diffs >= 0, "#648FFF", "#DC267F")
ax2.bar(residues, pct_diffs, color=colors, width=0.8, alpha=0.8)
ax2.axhline(0, color="black", lw=0.8)
ax2.axvspan(84,  90,  color="grey", alpha=0.3, label="Gate (r84–90)")
ax2.axvspan(114, 118, color="grey", alpha=0.3, label="Latch (r114–118)")

ax2.set_xlabel("Residue number", fontsize=9)
ax2.set_ylabel("% difference\n(binder − nonbinder) / nonbinder", fontsize=9)
ax2.set_title("Per-residue RMSF % difference", fontsize=10)
ax2.legend(fontsize=7,
           handles=[plt.Rectangle((0,0),1,1, color="#648FFF", alpha=0.8),
                    plt.Rectangle((0,0),1,1, color="#DC267F", alpha=0.8)],
           labels=["Binder higher", "Nonbinder higher"])
ax2.grid(True, alpha=0.4, axis="y")

plt.show()

# -------------------------------------------------------------------
# Optional: save outputs
# -------------------------------------------------------------------
# fig.savefig(os.path.join(base, "analysis", "rmsf_binder_vs_nonbinder.png"),dpi=300, bbox_inches="tight")
# df.to_csv(os.path.join(base, "analysis", "rmsf_binder_vs_nonbinder.csv"))

In [ ]:
# ----------------------------
# Regions of interest
# ----------------------------
rmsf_regions = {
    "Gate\n(r84–90)":         list(range(84, 91)),
    "Latch\n(r114–118)":      list(range(114, 119)),
    "Loop Lb7a5\n(r148–155)": list(range(148, 156)),
    "loop La1b1\n(r20-25)": list(range(20, 25)),
    "Q69":                    [69],
    "I134":                   [134],
    "Y23":                    [23],
    "K59":                    [59],
    "R79":                    [79],
    "I110":                   [110],
    "G163":                   [163],
}

region_labels = list(rmsf_regions.keys())

# ----------------------------
# Load RMSF data
# ----------------------------
rows    = []
skipped = []

for folder_name, seq_type in all_systems:
    xvg = os.path.join(rmsf_run_dir(folder_name, seq_type), "rmsf_PL.xvg")

    if not os.path.exists(xvg):
        skipped.append(folder_name)
        continue

    rmsf_x, rmsf_y = get_data(xvg)
    rmsf_x_int = rmsf_x.astype(int)
    rmsf_y_ang = rmsf_y * NM_TO_ANG

    row = {
        "folder":      folder_name,
        "seq_type":    seq_type,
        "group_label": GROUP_LABEL[seq_type],
    }

    for region_label, residues in rmsf_regions.items():
        idx  = np.isin(rmsf_x_int, residues)
        vals = rmsf_y_ang[idx]
        row[region_label] = np.mean(vals) if vals.size > 0 else np.nan

    rows.append(row)

if skipped:
    print(f"Skipped {len(skipped)} missing files: "
          f"{skipped[:5]}{'...' if len(skipped) > 5 else ''}")

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} sequences  " +
      "  ".join(f"{lbl}: {(df.group_label == lbl).sum()}" for lbl in GROUP_ORDER))

from matplotlib.patches import Patch
from scipy.stats import mannwhitneyu, gaussian_kde
from itertools import combinations

legend_handles = [
    Patch(facecolor=GROUP_COLOR[lbl], label=lbl, alpha=0.85)
    for lbl in GROUP_ORDER
]

# ----------------------------
# Statistical helpers
# ----------------------------
GROUP_ABBREV = {
    "Binder":         "B",
    "False Positive": "FP",
    "Low Confidence": "LC",
    "Fail Geometry":  "FG",
}

def p_label(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def compute_pairwise_bonferroni(df, region, groups):
    """
    Pairwise two-sided Mann-Whitney U tests with Bonferroni correction
    across all valid pairs within this region.
    Returns dict: (g1, g2) -> corrected p-value (or nan if insufficient data).
    """
    pairs = list(combinations(groups, 2))
    raw   = {}
    for g1, g2 in pairs:
        v1 = df.loc[df.group_label == g1, region].dropna().values
        v2 = df.loc[df.group_label == g2, region].dropna().values
        if len(v1) >= 2 and len(v2) >= 2:
            _, p = mannwhitneyu(v1, v2, alternative="two-sided")
            raw[(g1, g2)] = p
        else:
            raw[(g1, g2)] = np.nan

    n_valid = sum(1 for v in raw.values() if not np.isnan(v))
    return {
        k: min(v * n_valid, 1.0) if not np.isnan(v) else np.nan
        for k, v in raw.items()
    }

def add_sig_brackets(ax, group_positions, pval_dict, data_ymax, data_range):
    """
    Draw stacked significance brackets above a strip plot.
    Only draws brackets for pairs with corrected p < 0.05.
    Shorter-span pairs are placed lower to minimize crossings.
    """
    sig = [
        (g1, g2, p) for (g1, g2), p in pval_dict.items()
        if not np.isnan(p) and p < 0.05
    ]
    # Sort by x-span: shorter spans (adjacent groups) sit lower
    sig.sort(key=lambda t: abs(group_positions[t[1]] - group_positions[t[0]]))

    y_step = 0.11 * data_range
    tip_h  = 0.025 * data_range
    base_y = data_ymax + 0.06 * data_range

    for level, (g1, g2, p) in enumerate(sig):
        x1, x2 = group_positions[g1], group_positions[g2]
        y = base_y + level * y_step
        ax.plot(
            [x1, x1, x2, x2], [y, y + tip_h, y + tip_h, y],
            lw=0.8, color="black", clip_on=False,
        )
        ax.text(
            (x1 + x2) / 2, y + tip_h * 1.2, p_label(p),
            ha="center", va="bottom", fontsize=6, color="black",
        )

    # Expand y-axis to fit all brackets
    if sig:
        needed_top = base_y + len(sig) * y_step + y_step
        ax.set_ylim(top=max(ax.get_ylim()[1], needed_top))

# ----------------------------
# Figure 1: strip plot + mean bar + significance brackets
# ----------------------------
n_regions = len(region_labels)
fig1, axes1 = plt.subplots(
    1, n_regions,
    figsize=(3.2 * n_regions, 6.5),   # extra height for bracket stacking
    sharey=False,
    constrained_layout=True,
)

rng = np.random.default_rng(42)
group_positions = {g: i for i, g in enumerate(GROUP_ORDER)}

for ax, region in zip(axes1, region_labels):
    all_vals = df[region].dropna().values
    data_ymax  = all_vals.max()  if len(all_vals) > 0 else 1.0
    data_range = (all_vals.max() - all_vals.min()) if len(all_vals) > 1 else 1.0

    for x_pos, group_lbl in enumerate(GROUP_ORDER):
        vals  = df.loc[df.group_label == group_lbl, region].dropna().values
        color = GROUP_COLOR[group_lbl]
        if len(vals) == 0:
            continue

        jitter = rng.uniform(-0.18, 0.18, size=len(vals))
        ax.scatter(x_pos + jitter, vals,
                   color=color, alpha=0.55, s=16, linewidths=0, zorder=2)
        ax.plot([x_pos - 0.28, x_pos + 0.28], [np.mean(vals)] * 2,
                color="black", lw=2.0, solid_capstyle="round", zorder=3)

    ax.set_title(region, fontsize=9, pad=4)
    ax.set_xticks(range(len(GROUP_ORDER)))
    ax.set_xticklabels(GROUP_ORDER, fontsize=7, rotation=30, ha="right")
    ax.set_ylabel("Mean RMSF (Å)", fontsize=8)
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(True, axis="y", alpha=0.4, lw=0.6)
    ax.set_xlim(-0.6, len(GROUP_ORDER) - 0.4)

    pval_dict = compute_pairwise_bonferroni(df, region, GROUP_ORDER)
    add_sig_brackets(ax, group_positions, pval_dict, data_ymax, data_range)

fig1.legend(handles=legend_handles, loc="upper center", ncol=4,
            fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.04))
fig1.suptitle("RMSF by Region of Interest — All Sequences", fontsize=11, y=1.08)
plt.savefig(os.path.join(base, "analysis", "rmsf_regions_all_seqs.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# ----------------------------
# Figure 2: KDE distributions + significance annotation box
# ----------------------------
fig2, axes2 = plt.subplots(
    1, n_regions,
    figsize=(3.5 * n_regions, 4.5),
    sharey=False,
    constrained_layout=True,
)

for ax, region in zip(axes2, region_labels):
    x_min, x_max = np.inf, -np.inf
    group_data   = {}

    for group_lbl in GROUP_ORDER:
        vals = df.loc[df.group_label == group_lbl, region].dropna().values
        if len(vals) >= 2:
            group_data[group_lbl] = vals
            x_min = min(x_min, vals.min())
            x_max = max(x_max, vals.max())

    pad       = (x_max - x_min) * 0.15
    x_grid    = np.linspace(x_min - pad, x_max + pad, 300)
    y_max_all = 0

    for group_lbl, vals in group_data.items():
        color = GROUP_COLOR[group_lbl]
        kde   = gaussian_kde(vals, bw_method=0.5)
        y     = kde(x_grid)
        y_max_all = max(y_max_all, y.max())

        ax.plot(x_grid, y, color=color, lw=1.8)
        ax.fill_between(x_grid, y, alpha=0.15, color=color)
        ax.plot(vals, np.full_like(vals, -0.015 * y_max_all),
                "|", color=color, alpha=0.6, ms=6, mew=1.2)
        ax.axvline(np.median(vals), color=color, lw=1.0,
                   linestyle="--", alpha=0.7)

    # Significance annotation box (top-right corner)
    pval_dict = compute_pairwise_bonferroni(df, region, GROUP_ORDER)
    sig_lines = [
        f"{GROUP_ABBREV[g1]}/{GROUP_ABBREV[g2]}: {p_label(p)} (p={p:.3f})"
        for (g1, g2), p in pval_dict.items()
        if not np.isnan(p) and p < 0.05
    ]
    if sig_lines:
        ax.text(
            0.97, 0.97, "\n".join(sig_lines),
            transform=ax.transAxes,
            ha="right", va="top", fontsize=6,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.85),
        )

    ax.set_title(region, fontsize=9, pad=4)
    ax.set_xlabel("Mean RMSF (Å)", fontsize=8)
    ax.set_ylabel("Density", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, axis="y", alpha=0.4, lw=0.6)
    ax.set_xlim(x_min - pad, x_max + pad)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig2.legend(handles=legend_handles, loc="upper center", ncol=4,
            fontsize=8, frameon=False, bbox_to_anchor=(0.5, 1.04))
fig2.suptitle("RMSF Distributions by Region of Interest", fontsize=11, y=1.08)
plt.savefig(os.path.join(base, "analysis", "rmsf_regions_distributions.png"),
            dpi=300, bbox_inches="tight")
plt.show()

In [10]:
single_residue_labels = ["Q69", "I134", "Y23", "K59", "R79", "I110", "G163"]

# These columns are already in df as the region label (e.g. "Q69")
# Rename to match the multi-residue CSV style for consistency
rename_map = {r: f"{r} RMSF (A)" for r in single_residue_labels}

single_res_df = (
    df[["folder", "group_label"] + single_residue_labels]
    .rename(columns={"folder": "Sequence", "group_label": "Group"} | rename_map)
    .reset_index(drop=True)
)

print(single_res_df.to_string())

single_res_df.to_csv(
    os.path.join(base, "analysis", "rmsf_single_residues_per_seq.csv"),
    index=False,
)

                Sequence           Group  Q69 RMSF (A)  I134 RMSF (A)  Y23 RMSF (A)  K59 RMSF (A)  R79 RMSF (A)  I110 RMSF (A)  G163 RMSF (A)
0           seq16_binder          Binder         3.413          2.474         2.893         0.759         0.546          0.676          0.759
1           seq14_binder          Binder         3.025          2.581         2.463         0.664         0.510          0.684          0.620
2           seq10_binder          Binder         2.972          2.772         1.674         0.815         0.805          0.871          1.005
3           seq17_binder          Binder         2.681          2.556         1.387         0.667         0.573          0.682          0.598
4           seq39_binder          Binder         3.807          2.558         2.160         0.724         0.663          0.671          1.319
5           seq42_binder          Binder         3.145          2.147         2.926         0.838         1.628          0.947          1.015
6     

In [ ]:
# -------------------------------------------------------------------
# Per-residue Cα RMSF for structural regions
# Binders vs all nonbinders (nb + low_pkt + fail_gate collapsed)
# Assumes loading cell has already run (base, runrel, TYPE_SUBDIR,
# binder_list, nonbinder_list, GROUP_COLOR, NM_TO_ANG, rmsf_run_dir)
# -------------------------------------------------------------------

REGIONS = {
    "Gate (r84-90)":      "rmsf_PL_ca_gate.xvg",
    "Latch (r114-118)":   "rmsf_PL_ca_latch.xvg",
    "Lβ7α5 (r148-155)":  "rmsf_PL_ca_Lb7a5.xvg",
    "Recoil (r154-166)":  "rmsf_PL_ca_recoil.xvg",
}

COLOR_B  = GROUP_COLOR["Binder"]
COLOR_NB = GROUP_COLOR["False Positive"]   # representative color for collapsed nonbinders

# -------------------------------------------------------------------
# Load per-residue RMSF for a list of (folder_name, seq_type) tuples
# Returns {residue_id: [rmsf_seq1, rmsf_seq2, ...]} in Å
# -------------------------------------------------------------------
def load_rmsf_group(seq_list, region_fname):
    group = {}
    for folder_name, seq_type in seq_list:
        filepath = os.path.join(rmsf_run_dir(folder_name, seq_type), region_fname)
        if not os.path.exists(filepath):
            print(f"WARNING: not found — {filepath}")
            continue
        x_data, y_data = get_data(filepath)
        for resid, rmsf in zip(x_data.astype(int), y_data):
            group.setdefault(resid, []).append(rmsf * NM_TO_ANG)
    return group

# -------------------------------------------------------------------
# Build per-region summary DataFrames and plot
# -------------------------------------------------------------------
n_panels = len(REGIONS)
fig, axes = plt.subplots(
    nrows=n_panels, ncols=1,
    figsize=(12, 4 * n_panels), dpi=150,
    constrained_layout=True,
)
if n_panels == 1:
    axes = [axes]

for ax, (region_name, region_fname) in zip(axes, REGIONS.items()):

    binder_data    = load_rmsf_group(binder_list,    region_fname)
    nonbinder_data = load_rmsf_group(nonbinder_list, region_fname)

    common_residues = sorted(set(binder_data.keys()) & set(nonbinder_data.keys()))

    rows = []
    for resid in common_residues:
        b_vals  = binder_data[resid]
        nb_vals = nonbinder_data[resid]

        b_mean   = np.mean(b_vals)
        b_sd     = np.std(b_vals,  ddof=1) if len(b_vals)  > 1 else 0.0
        nb_mean  = np.mean(nb_vals)
        nb_sd    = np.std(nb_vals, ddof=1) if len(nb_vals) > 1 else 0.0
        pct_diff = (b_mean - nb_mean) / nb_mean * 100 if nb_mean != 0 else np.nan

        rows.append({
            "Residue":            resid,
            "Binder mean (A)":    round(b_mean,            6),
            "Binder SD (A)":      round(b_sd,              6),
            "Nonbinder mean (A)": round(nb_mean,            6),
            "Nonbinder SD (A)":   round(nb_sd,              6),
            "Diff (B - NB, A)":   round(b_mean - nb_mean,   6),
            "% diff (vs NB)":     round(pct_diff,            4),
        })

    df = pd.DataFrame(rows).set_index("Residue")
    print(f"\n{'='*60}")
    print(f"  {region_name}  |  "
          f"Binders: n={len(binder_list)}  Nonbinders: n={len(nonbinder_list)}")
    print(f"{'='*60}")
    print(df.to_string())

    residues = np.array(common_residues)
    b_means  = df["Binder mean (A)"].values
    b_sds    = df["Binder SD (A)"].values
    nb_means = df["Nonbinder mean (A)"].values
    nb_sds   = df["Nonbinder SD (A)"].values

    ax.plot(residues, b_means,  color=COLOR_B,  lw=1.5,
            label=f"Binders (n={len(binder_list)}, mean)")
    ax.fill_between(residues, b_means - b_sds, b_means + b_sds,
                    color=COLOR_B,  alpha=0.2, label="Binders ± SD")

    ax.plot(residues, nb_means, color=COLOR_NB, lw=1.5,
            label=f"Nonbinders (n={len(nonbinder_list)}, mean)")
    ax.fill_between(residues, nb_means - nb_sds, nb_means + nb_sds,
                    color=COLOR_NB, alpha=0.2, label="Nonbinders ± SD")

    ax.set_title(f"{region_name} Cα RMSF: binders vs nonbinders", fontsize=10)
    ax.set_xlabel("Residue number", fontsize=9)
    ax.set_ylabel("RMSF (Å)", fontsize=9)
    ax.set_xticks(residues)
    ax.set_xticklabels(residues, fontsize=8)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.4)

plt.show()

# fig.savefig(os.path.join(base, "analysis", "rmsf_ca_regions_binders_vs_nonbinders.png"), dpi=300, bbox_inches="tight")

In [ ]:
# -------------------------------------------------------------------
# Cα RMSF per-sequence summary table + region comparison plots
# Assumes loading cell has already run:
#   base, runrel, TYPE_SUBDIR, GROUP_LABEL, GROUP_COLOR,
#   GROUP_ORDER, all_systems, binder_list, nonbinder_list, NM_TO_ANG,
#   rmsf_run_dir
# -------------------------------------------------------------------
from matplotlib.patches import Patch

# Plot-specific constants (not in loading cell)
POOLED_COLOR = {
    "Binder":    GROUP_COLOR["Binder"],
    "Nonbinder": GROUP_COLOR["False Positive"],
}

REGIONS = {
    "Gate (r84-90)":             "rmsf_PL_ca_gate.xvg",
    "Latch (r114-118)":          "rmsf_PL_ca_latch.xvg",
    "Lβ7α5 (r148-155)":         "rmsf_PL_ca_Lb7a5.xvg",
    "C-term α-helix (r154-166)": "rmsf_PL_ca_recoil.xvg",
}

# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------
def truncate_label(folder_name):
    parts = folder_name.split("_")
    if parts[0] in ("pair", "bind", "nonb") and len(parts) >= 2:
        return parts[1]                       # pair_0001_binder  → 0001
                                                # bind_019_binder   → 019
                                                # nonb_006_nb       → 006
    if folder_name.startswith("seq"):
        return folder_name.split("_")[0][3:]  # seq16_binder      → 16
    return folder_name

def load_rmsf_single(folder_name, seq_type, region_fname):
    filepath = os.path.join(rmsf_run_dir(folder_name, seq_type), region_fname)
    if not os.path.exists(filepath):
        print(f"WARNING: not found — {filepath}")
        return []
    _, rmsf = get_data(filepath)
    return (rmsf * NM_TO_ANG).tolist()

# -------------------------------------------------------------------
# Build per-sequence DataFrame
# all_systems: (folder_name, seq_type)
# -------------------------------------------------------------------
rows = []
for folder_name, seq_type in all_systems:
    group = GROUP_LABEL[seq_type]
    row   = {"Sequence": folder_name, "Group": group}
    for region_name, region_fname in REGIONS.items():
        vals = load_rmsf_single(folder_name, seq_type, region_fname)
        row[f"{region_name} mean (A)"] = round(np.mean(vals),        5) if vals          else np.nan
        row[f"{region_name} SD (A)"]   = round(np.std(vals, ddof=1), 5) if len(vals) > 1 else np.nan
    rows.append(row)

df = pd.DataFrame(rows).set_index("Sequence")
print(df.to_string())

# -------------------------------------------------------------------
# Per-sequence colors — flat group color for every sequence
# -------------------------------------------------------------------
seq_colors = {}
group_seqs = {}   # group_label → [folder_name, ...]
for folder_name, seq_type in all_systems:
    group = GROUP_LABEL[seq_type]
    group_seqs.setdefault(group, []).append(folder_name)

for group, seqs in group_seqs.items():
    for folder_name in seqs:
        seq_colors[folder_name] = GROUP_COLOR[group]

group_sizes   = [len(group_seqs.get(g, [])) for g in GROUP_ORDER]
group_offsets = np.cumsum([0] + group_sizes[:-1])
region_names  = list(REGIONS.keys())

# -------------------------------------------------------------------
# Figure 1: per-sequence bars, 2×2 grid
# -------------------------------------------------------------------
fig1, axes1 = plt.subplots(2, 2, figsize=(16, 10), dpi=150, sharey=False)
axes1 = axes1.flatten()

for ax, region_name in zip(axes1, region_names):
    seqs        = df.index.tolist()
    tick_labels = [truncate_label(s) for s in seqs]
    means       = df[f"{region_name} mean (A)"].values
    sds         = df[f"{region_name} SD (A)"].values
    colors      = [seq_colors[s] for s in seqs]
    x           = np.arange(len(seqs))

    ax.bar(x, means, yerr=sds, color=colors, capsize=4,
           edgecolor="white", linewidth=0.5)

    for offset in group_offsets[1:]:
        ax.axvline(offset - 0.5, color="gray", linewidth=1.0,
                   linestyle="--", alpha=0.7)

    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=9)
    ax.set_title(region_name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Sequence ID", fontsize=10)
    ax.set_ylabel("Mean RMSF (Å)", fontsize=10)
    ax.grid(True, axis="y", alpha=0.35)
    ax.set_axisbelow(True)

legend_handles1 = [Patch(facecolor=GROUP_COLOR[g], label=g) for g in GROUP_ORDER]
fig1.legend(handles=legend_handles1,
            loc="center left", bbox_to_anchor=(1.01, 0.5),
            ncol=1, fontsize=10, frameon=True,
            title="Group", title_fontsize=11)
fig1.suptitle("Mean Cα RMSF per Region by Sequence", fontsize=14, fontweight="bold")
plt.tight_layout()

# -------------------------------------------------------------------
# Figure 2: group-average bars, 2×2 grid
# -------------------------------------------------------------------
group_summary = {}
for group in GROUP_ORDER:
    seqs_in_group = group_seqs.get(group, [])
    group_summary[group] = {}
    for region_name in region_names:
        col  = f"{region_name} mean (A)"
        vals = df.loc[df.index.isin(seqs_in_group), col].dropna().values
        group_summary[group][region_name] = {
            "mean": np.mean(vals)        if len(vals) > 0 else np.nan,
            "sd":   np.std(vals, ddof=1) if len(vals) > 1 else np.nan,
        }

fig2, axes2 = plt.subplots(2, 2, figsize=(12, 8), dpi=150, sharey=False)
axes2 = axes2.flatten()

x2      = np.arange(len(GROUP_ORDER))
colors2 = [GROUP_COLOR[g] for g in GROUP_ORDER]

for ax, region_name in zip(axes2, region_names):
    means2 = [group_summary[g][region_name]["mean"] for g in GROUP_ORDER]
    sds2   = [group_summary[g][region_name]["sd"]   for g in GROUP_ORDER]

    ax.bar(x2, means2, yerr=sds2, color=colors2, capsize=5,
           edgecolor="white", linewidth=0.5, width=0.55)
    ax.set_xticks(x2)
    ax.set_xticklabels(GROUP_ORDER, fontsize=10)
    ax.set_title(region_name, fontsize=12, fontweight="bold")
    ax.set_ylabel("Mean RMSF (Å)", fontsize=10)
    ax.grid(True, axis="y", alpha=0.35)
    ax.set_axisbelow(True)

legend_handles2 = [Patch(facecolor=GROUP_COLOR[g], label=g) for g in GROUP_ORDER]
fig2.legend(handles=legend_handles2,
            loc="center left", bbox_to_anchor=(1.01, 0.5),
            ncol=1, fontsize=10, frameon=True,
            title="Group", title_fontsize=11)
fig2.suptitle("Group-average Cα RMSF per Region", fontsize=14, fontweight="bold")
plt.tight_layout()

# -------------------------------------------------------------------
# Figure 3: pooled Binder vs. Nonbinder, 2×2 grid
# -------------------------------------------------------------------
NONBINDER_GROUPS = [g for g in GROUP_ORDER if g != "Binder"]
pooled_seqs = {
    "Binder":    group_seqs.get("Binder", []),
    "Nonbinder": [s for g in NONBINDER_GROUPS for s in group_seqs.get(g, [])],
}
pooled_order = ["Binder", "Nonbinder"]

pooled_summary = {}
for pool_label, seqs_in_pool in pooled_seqs.items():
    pooled_summary[pool_label] = {}
    for region_name in region_names:
        col  = f"{region_name} mean (A)"
        vals = df.loc[df.index.isin(seqs_in_pool), col].dropna().values
        pooled_summary[pool_label][region_name] = {
            "mean": np.mean(vals)        if len(vals) > 0 else np.nan,
            "sd":   np.std(vals, ddof=1) if len(vals) > 1 else np.nan,
            "n":    len(vals),
        }

fig3, axes3 = plt.subplots(2, 2, figsize=(10, 7), dpi=150, sharey=False)
axes3 = axes3.flatten()

x3      = np.arange(len(pooled_order))
colors3 = [POOLED_COLOR[g] for g in pooled_order]

for ax, region_name in zip(axes3, region_names):
    means3 = [pooled_summary[g][region_name]["mean"] for g in pooled_order]
    sds3   = [pooled_summary[g][region_name]["sd"]   for g in pooled_order]
    ns     = [pooled_summary[g][region_name]["n"]    for g in pooled_order]

    bars = ax.bar(x3, means3, yerr=sds3, color=colors3, capsize=6,
                  edgecolor="white", linewidth=0.5, width=0.5)

    for bar, sd, n in zip(bars, sds3, ns):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (sd or 0) + 0.01,
                f"n={n}", ha="center", va="bottom", fontsize=9, color="#444444")

    ax.set_xticks(x3)
    ax.set_xticklabels(pooled_order, fontsize=11)
    ax.set_title(region_name, fontsize=12, fontweight="bold")
    ax.set_ylabel("Mean RMSF (Å)", fontsize=10)
    ax.grid(True, axis="y", alpha=0.35)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

legend_handles3 = [Patch(facecolor=POOLED_COLOR[g], label=g) for g in pooled_order]
fig3.legend(handles=legend_handles3,
            loc="center left", bbox_to_anchor=(1.01, 0.5),
            ncol=1, fontsize=11, frameon=True,
            title="Group", title_fontsize=12)
fig3.suptitle(
    "Binder vs. Nonbinder (pooled)\nGroup-average Cα RMSF per Region",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

# fig1.savefig(os.path.join(base, "analysis", "rmsf_ca_per_seq_per_region.png"),        dpi=300, bbox_inches="tight")
# fig2.savefig(os.path.join(base, "analysis", "rmsf_ca_group_avg_per_region.png"),       dpi=300, bbox_inches="tight")
# fig3.savefig(os.path.join(base, "analysis", "rmsf_ca_binder_vs_nonbinder_pooled.png"), dpi=300, bbox_inches="tight")
# df.to_csv(os.path.join(base, "analysis", "rmsf_ca_per_seq_summary.csv"))

In [16]:
# -------------------------------------------------------------------
# Generate 250ns equivalents of:
#   rmsf_single_residues_per_seq.csv  →  rmsf_single_residues_per_seq_250ns.csv
#   rmsf_ca_per_seq_summary.csv       →  rmsf_ca_per_seq_summary_250ns.csv
#
# Files are read from half_time/ under each sequence's runrel directory.
# Requires loading cell to have already run:
#   base, runrel, TYPE_SUBDIR, GROUP_LABEL, GROUP_ORDER, all_systems, NM_TO_ANG
# -------------------------------------------------------------------

HALF_TIME = "half_time"   # subdirectory under runrel

# ── Helper: load from half_time/ ──────────────────────────────────
def load_rmsf_250ns(folder_name, seq_type, fname):
    """Load an RMSF .xvg from the half_time/ subdirectory.
    Returns (x_int, y_ang) arrays, or (None, None) if missing."""
    filepath = os.path.join(
        base, TYPE_SUBDIR[seq_type], folder_name,
        runrel, HALF_TIME, fname
    )
    if not os.path.exists(filepath):
        print(f"WARNING: not found — {filepath}")
        return None, None
    x, y = get_data(filepath)
    return x.astype(int), y * NM_TO_ANG


# ════════════════════════════════════════════════════════════════════
# 1. rmsf_single_residues_per_seq_250ns.csv
#    Source: half_time/rmsf_PL_250ns.xvg (full per-residue RMSF)
# ════════════════════════════════════════════════════════════════════
rmsf_regions_250 = {
    "Gate\n(r84–90)":         list(range(84, 91)),
    "Latch\n(r114–118)":      list(range(114, 119)),
    "Loop Lb7a5\n(r148–155)": list(range(148, 156)),
    "loop La1b1\n(r20-25)":   list(range(20, 25)),
    "Q69":                    [69],
    "I134":                   [134],
    "Y23":                    [23],
    "K59":                    [59],
    "R79":                    [79],
    "I110":                   [110],
    "G163":                   [163],
}
single_residue_labels_250 = ["Q69", "I134", "Y23", "K59", "R79", "I110", "G163"]

rows_single = []
skipped_single = []

for folder_name, seq_type in all_systems:
    rmsf_x, rmsf_y = load_rmsf_250ns(folder_name, seq_type, "rmsf_PL_250ns.xvg")
    if rmsf_x is None:
        skipped_single.append(folder_name)
        continue

    row = {
        "folder":      folder_name,
        "seq_type":    seq_type,
        "group_label": GROUP_LABEL[seq_type],
    }
    for region_label, residues in rmsf_regions_250.items():
        idx  = np.isin(rmsf_x, residues)
        vals = rmsf_y[idx]
        row[region_label] = np.mean(vals) if vals.size > 0 else np.nan
    rows_single.append(row)

if skipped_single:
    print(f"rmsf_PL_250ns.xvg missing for {len(skipped_single)} sequences: "
          f"{skipped_single[:5]}{'...' if len(skipped_single) > 5 else ''}")

df_single_250 = pd.DataFrame(rows_single)

rename_map_250 = {r: f"{r} RMSF (A)" for r in single_residue_labels_250}
single_res_250_df = (
    df_single_250[["folder", "group_label"] + single_residue_labels_250]
    .rename(columns={"folder": "Sequence", "group_label": "Group"} | rename_map_250)
    .reset_index(drop=True)
)

csv_path_single = os.path.join(base, "analysis", "rmsf_single_residues_per_seq_250ns.csv")
single_res_250_df.to_csv(csv_path_single, index=False)
print(f"Saved ({len(single_res_250_df)} rows) → {csv_path_single}")
print(single_res_250_df.to_string())


# ════════════════════════════════════════════════════════════════════
# 2. rmsf_ca_per_seq_summary_250ns.csv
#    Source: half_time/rmsf_PL_ca_{gate,latch,Lb7a5,recoil}_250ns.xvg
# ════════════════════════════════════════════════════════════════════
REGIONS_250 = {
    "Gate (r84-90)":             "rmsf_PL_ca_gate_250ns.xvg",
    "Latch (r114-118)":          "rmsf_PL_ca_latch_250ns.xvg",
    "Lβ7α5 (r148-155)":         "rmsf_PL_ca_Lb7a5_250ns.xvg",
    "C-term α-helix (r154-166)": "rmsf_PL_ca_recoil_250ns.xvg",
}

rows_ca = []
for folder_name, seq_type in all_systems:
    group = GROUP_LABEL[seq_type]
    row   = {"Sequence": folder_name, "Group": group}
    for region_name, region_fname in REGIONS_250.items():
        _, vals = load_rmsf_250ns(folder_name, seq_type, region_fname)
        if vals is not None and len(vals) > 0:
            row[f"{region_name} mean (A)"] = round(float(np.mean(vals)),        5)
            row[f"{region_name} SD (A)"]   = round(float(np.std(vals, ddof=1)), 5) if len(vals) > 1 else np.nan
        else:
            row[f"{region_name} mean (A)"] = np.nan
            row[f"{region_name} SD (A)"]   = np.nan
    rows_ca.append(row)

df_ca_250 = pd.DataFrame(rows_ca).set_index("Sequence")

csv_path_ca = os.path.join(base, "analysis", "rmsf_ca_per_seq_summary_250ns.csv")
df_ca_250.to_csv(csv_path_ca)
print(f"\nSaved ({len(df_ca_250)} rows) → {csv_path_ca}")
print(df_ca_250.to_string())

Saved (130 rows) → /Users/ivanatang/Library/CloudStorage/OneDrive-UCB-O365/Shirts Lab/LCA_boltz_models/analysis/rmsf_single_residues_per_seq_250ns.csv
                Sequence           Group  Q69 RMSF (A)  I134 RMSF (A)  Y23 RMSF (A)  K59 RMSF (A)  R79 RMSF (A)  I110 RMSF (A)  G163 RMSF (A)
0           seq16_binder          Binder         3.310          2.381         1.289         0.582         0.512          0.639          0.617
1           seq14_binder          Binder         3.340          2.477         2.905         0.685         0.530          0.677          0.644
2           seq10_binder          Binder         3.418          2.659         1.609         0.796         0.718          0.878          0.979
3           seq17_binder          Binder         2.535          2.581         1.361         0.695         0.636          0.631          0.485
4           seq39_binder          Binder         3.802          2.297         2.392         0.707         0.596          0.680          0.9